# Module 5 Homework

In this homework we'll put what we learned about Spark in practice.

For this homework we will be using the Yellow 2024-10 data from the official website: 

```bash
wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-10.parquet
```


## Question 1: Install Spark and PySpark

- Install Spark
- Run PySpark
- Create a local spark session
- Execute spark.version.

What's the output?


In [1]:
!pyspark --version

Welcome to
      ____              __
     / __/__  ___ _____/ /__
    _\ \/ _ \/ _ `/ __/  '_/
   /___/ .__/\_,_/_/ /_/\_\   version 3.3.2
      /_/
                        
Using Scala version 2.12.15, OpenJDK 64-Bit Server VM, 11.0.2
Branch HEAD
Compiled by user liangchi on 2023-02-10T19:57:40Z
Revision 5103e00c4ce5fcc4264ca9c4df12295d42557af6
Url https://github.com/apache/spark
Type --help for more information.


> [!NOTE]
> To install PySpark follow this [guide](https://github.com/DataTalksClub/data-engineering-zoomcamp/blob/main/05-batch/setup/pyspark.md)


## Question 2: Yellow October 2024

Read the October 2024 Yellow into a Spark Dataframe.

Repartition the Dataframe to 4 partitions and save it to parquet.

What is the average size of the Parquet (ending with .parquet extension) Files that were created (in MB)? Select the answer which most closely matches.
==***(Correct!)***==

- 6MB
- 25MB
- 75MB
- 100MB


In [30]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import types
from pyspark.sql.functions import col, to_date, count, asc



In [3]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


25/03/13 12:25:13 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
spark.version

'3.3.2'

In [5]:
!ls -lh homework_data/yellow_tripdata_2024-10.parquet


-rw-rw-r-- 1 alvarovs alvarovs 62M Dec 18 21:21 homework_data/yellow_tripdata_2024-10.parquet


In [6]:
df_yellow = spark.read.parquet('homework_data/yellow_tripdata_2024-10.parquet')


In [7]:
df_yellow.printSchema()


root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)



In [10]:
df_yellow_partitioned = df_yellow.repartition(4)

df_yellow_partitioned.write.parquet('homework_data/partitioned/2024/10/')

In [11]:
!ls -lh homework_data/partitioned/2024/10/


total 97M
-rw-r--r-- 1 alvarovs alvarovs   0 Mar 13 12:29 _SUCCESS
-rw-r--r-- 1 alvarovs alvarovs 25M Mar 13 12:29 part-00000-985a0609-9970-44c6-8133-a258f48ddca0-c000.snappy.parquet
-rw-r--r-- 1 alvarovs alvarovs 25M Mar 13 12:29 part-00001-985a0609-9970-44c6-8133-a258f48ddca0-c000.snappy.parquet
-rw-r--r-- 1 alvarovs alvarovs 25M Mar 13 12:29 part-00002-985a0609-9970-44c6-8133-a258f48ddca0-c000.snappy.parquet
-rw-r--r-- 1 alvarovs alvarovs 25M Mar 13 12:29 part-00003-985a0609-9970-44c6-8133-a258f48ddca0-c000.snappy.parquet


In [20]:
# Convert the timestamp column to date format
df = df_yellow.withColumn("pickup_date", to_date(col("tpep_pickup_datetime")))

# Filter trips that started on October 15
oct_15_trips = df.filter(col("pickup_date") == "2024-10-15")  # Change year if needed

# Count the number of trips
trip_count = oct_15_trips.count()

# Show the result
print(f"Number of trips on October 15: {trip_count}")

Number of trips on October 15: 128893


In [24]:
from pyspark.sql.functions import col, unix_timestamp, max

In [26]:
# Compute trip duration in hours
df = df.withColumn("trip_duration_hours", 
                   (unix_timestamp(col("tpep_dropoff_datetime")) - unix_timestamp(col("tpep_pickup_datetime"))) / 3600)

# Find the longest trip duration
longest_trip = df.agg(max("trip_duration_hours")).collect()[0][0]

# Show the result
print(f"Longest trip duration: {longest_trip:.2f} hours")

Longest trip duration: 162.62 hours


In [27]:
# Load the Taxi Zone Lookup data
zones_df = spark.read.option("header", "true").csv("taxi_zone_lookup.csv", inferSchema=True)

# Join taxi data with zone lookup data on PULocationID
joined_df = df_yellow.join(zones_df, df_yellow.PULocationID == zones_df.LocationID, "left")


In [31]:

# Count pickups per zone
pickup_counts = joined_df.groupBy("Zone").agg(count("PULocationID").alias("pickup_count"))

# Find the least frequent pickup zone
least_frequent_zone = pickup_counts.orderBy(asc("pickup_count")).limit(1)

# Show the result
least_frequent_zone.show()

[Stage 25:>                                                         (0 + 4) / 4]

+--------------------+------------+
|                Zone|pickup_count|
+--------------------+------------+
|Governor's Island...|           1|
+--------------------+------------+

